In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)

# Load league-wide game data and filter to ATL home games
data = pd.read_csv('../../data/league_weather_2021_2025.csv')
atl = data[data['home_team'] == 'ATL'].copy()

atl['game_date'] = pd.to_datetime(atl['game_date'])
atl['temp_bin'] = pd.qcut(atl['temp_f'], q=5, duplicates='drop')
atl['rhum_bin'] = pd.qcut(atl['rhum'], q=5, duplicates='drop')

print(f'Total ATL home games: {len(atl)}')
print('\nTemperature bins and game counts:')
print(atl['temp_bin'].value_counts().sort_index())
print('\nHumidity bins and game counts:')
print(atl['rhum_bin'].value_counts().sort_index())

In [ ]:
# ============================================================
# Summary Tables
# ============================================================
temp_summary = atl.groupby('temp_bin', observed=True).agg(
    games=('away_runs_scored', 'size'),
    away_runs_mean=('away_runs_scored', 'mean'),
    away_runs_std=('away_runs_scored', 'std'),
    away_runs_median=('away_runs_scored', 'median'),
    temp_mean=('temp_f', 'mean'),
    rhum_mean=('rhum', 'mean'),
    pres_mean=('pres', 'mean'),
    wspd_mean=('wspd_mph', 'mean'),
).round(2)
temp_summary.index.name = 'Temperature Bin (degF)'
temp_summary.columns = ['Games', 'Away Runs Mean', 'Away Runs Std', 'Away Runs Median', 'Temp Mean', 'Humidity Mean', 'Pressure Mean', 'Wind Mean']

rhum_summary = atl.groupby('rhum_bin', observed=True).agg(
    games=('away_runs_scored', 'size'),
    away_runs_mean=('away_runs_scored', 'mean'),
    away_runs_std=('away_runs_scored', 'std'),
    away_runs_median=('away_runs_scored', 'median'),
    temp_mean=('temp_f', 'mean'),
    rhum_mean=('rhum', 'mean'),
    pres_mean=('pres', 'mean'),
    wspd_mean=('wspd_mph', 'mean'),
).round(2)
rhum_summary.index.name = 'Humidity Bin (%)'
rhum_summary.columns = ['Games', 'Away Runs Mean', 'Away Runs Std', 'Away Runs Median', 'Temp Mean', 'Humidity Mean', 'Pressure Mean', 'Wind Mean']

print('Temperature summary:')
display(temp_summary)

print('Humidity summary:')
display(rhum_summary)

In [ ]:
# ============================================================
# Boxplots: Spread of Away Runs by Bracket
# ============================================================
temp_order = atl['temp_bin'].cat.categories
temp_labels = [str(b).replace('(', '').replace(']', '').replace(', ', ' to ') for b in temp_order]
temp_box_data = [atl.loc[atl['temp_bin'] == b, 'away_runs_scored'].dropna().values for b in temp_order]

rhum_order = atl['rhum_bin'].cat.categories
rhum_labels = [str(b).replace('(', '').replace(']', '').replace(', ', ' to ') for b in rhum_order]
rhum_box_data = [atl.loc[atl['rhum_bin'] == b, 'away_runs_scored'].dropna().values for b in rhum_order]

fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharey=True)

bp1 = axes[0].boxplot(temp_box_data, patch_artist=True, labels=temp_labels, medianprops=dict(color='black', linewidth=1.5))
for patch in bp1['boxes']:
    patch.set(facecolor='#d62728', alpha=0.55)
axes[0].set_title('Away Runs Spread by Temperature Bracket')
axes[0].set_xlabel('Temperature Bracket (degF)')
axes[0].set_ylabel('Away Runs Scored')
axes[0].tick_params(axis='x', rotation=20)
axes[0].grid(axis='y', alpha=0.3)

bp2 = axes[1].boxplot(rhum_box_data, patch_artist=True, labels=rhum_labels, medianprops=dict(color='black', linewidth=1.5))
for patch in bp2['boxes']:
    patch.set(facecolor='#1f77b4', alpha=0.55)
axes[1].set_title('Away Runs Spread by Humidity Bracket')
axes[1].set_xlabel('Humidity Bracket (%)')
axes[1].tick_params(axis='x', rotation=20)
axes[1].grid(axis='y', alpha=0.3)

fig.suptitle('Truist Park: Distribution of Away Runs Across Weather Brackets', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Mean and IQR Overlay View
# ============================================================
def iqr_bounds(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    return q1, q3

temp_grouped = atl.groupby('temp_bin', observed=True)['away_runs_scored']
temp_means = temp_grouped.mean().reindex(temp_order)
temp_q1 = temp_grouped.quantile(0.25).reindex(temp_order)
temp_q3 = temp_grouped.quantile(0.75).reindex(temp_order)

rhum_grouped = atl.groupby('rhum_bin', observed=True)['away_runs_scored']
rhum_means = rhum_grouped.mean().reindex(rhum_order)
rhum_q1 = rhum_grouped.quantile(0.25).reindex(rhum_order)
rhum_q3 = rhum_grouped.quantile(0.75).reindex(rhum_order)

fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)

x_temp = np.arange(len(temp_order))
axes[0].plot(x_temp, temp_means, marker='o', color='#d62728', linewidth=2)
axes[0].fill_between(x_temp, temp_q1, temp_q3, color='#d62728', alpha=0.2)
axes[0].set_xticks(x_temp)
axes[0].set_xticklabels(temp_labels, rotation=20, ha='right')
axes[0].set_title('Away Runs Mean and IQR by Temperature Bracket')
axes[0].set_xlabel('Temperature Bracket (degF)')
axes[0].set_ylabel('Away Runs Scored')
axes[0].grid(True, alpha=0.3)

x_rhum = np.arange(len(rhum_order))
axes[1].plot(x_rhum, rhum_means, marker='o', color='#1f77b4', linewidth=2)
axes[1].fill_between(x_rhum, rhum_q1, rhum_q3, color='#1f77b4', alpha=0.2)
axes[1].set_xticks(x_rhum)
axes[1].set_xticklabels(rhum_labels, rotation=20, ha='right')
axes[1].set_title('Away Runs Mean and IQR by Humidity Bracket')
axes[1].set_xlabel('Humidity Bracket (%)')
axes[1].grid(True, alpha=0.3)

fig.suptitle('Truist Park: Central Tendency and Spread of Away Runs', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()